# 05 · Training setup (Step 3.1)

**Runtime: L4 GPU** (Runtime → Change runtime type → **L4 GPU**).

Checks everything training needs before any real run:
1. **W&B login** with the key in Colab Secrets (`WANDB_API_KEY`).
2. **QLoRA smoke test** (200 examples, 30 steps): the run is stopped on purpose right after its step-20 checkpoint, like a lost Colab connection, then **resumed**; it must continue at step 20 and finish at 30.
3. **LoRA (bf16) smoke test**: the main training mode, 30 steps.
4. **Memory and speed profile** at 3 image sizes × 3 batch sizes, to fix the image size for all training.

Expect about **40–60 minutes**. Every command runs through `sh()`, which stops with an error if a command fails (cell 5 expects one failure, on purpose).

## 1 · Settings, Drive, code, W&B key

In [ ]:
import json
import os
import subprocess

from google.colab import drive, userdata

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
LOCAL = "/content/data"
CKPT = f"{DRIVE_ROOT}/checkpoints"
LOGS = f"{DRIVE_ROOT}/logs"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"


def sh(cmd, check=True, **env):
    """Run a shell command, stream its output, and raise if it fails (unless check=False)."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"Command failed (exit {code}). Stop here and paste the output above.")
    return code


try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")  # never printed
    print("W&B key loaded from Colab Secrets.")
except Exception as e:
    raise RuntimeError(
        "Add WANDB_API_KEY in the key icon (Secrets) on the left, turn on notebook access, then rerun."
    ) from e
os.environ["WANDB_PROJECT"] = "vizwiz-assistant"
os.environ["HF_HOME"] = "/content/hf_cache"

drive.mount("/content/drive")
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

**Paste back:** the commit line and the GPU line (should say NVIDIA L4).

## 2 · Pinned environment (~3–5 min)
Now also installs bitsandbytes (QLoRA) and wandb.

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e {REPO_DIR} -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")
sh(f"uv pip freeze --python {PY} | grep -iE '^(bitsandbytes|wandb)=='")

**Paste back:** the `bitsandbytes==` and `wandb==` lines.

## 3 · W&B login check

In [ ]:
sh(f"{PY} -m vizwiz.train.wandb_check")

**Paste back:** the `W&B OK: ...` line. You can open the link to see the test run in your W&B project.

## 4 · Unzip training photos (~2–3 min)
Train and val are both carved from the official train split, so only `train.zip` is needed. The training script checks the label files against the committed hashes itself.

In [ ]:
%%time
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits train")

## 5 · QLoRA smoke test, stopped on purpose at step 20 (~5–8 min)
The first model download (~9 GB) happens here. The run **must** end with `--crash-at-step: stopping after the checkpoint at step 20`; the cell checks that and lists the saved checkpoints.

In [ ]:
%%time
SMOKE = "--set limit=200 --set max_steps=30 --set save_steps=10 --set eval_steps=1000 --set eval_limit=50 --set logging_steps=5"
TRAIN = f"{PY} -m vizwiz.train.sft train --config {REPO_DIR}/configs/train/answer.yaml --records {DATA}/processed --images {LOCAL}/images --out {CKPT} --wandb --wandb-group smoke"
sh(f"rm -rf {CKPT}/smoke-qlora {CKPT}/smoke-lora")  # fresh start if this notebook is rerun
code = sh(f"{TRAIN} --mode qlora --run-id smoke-qlora {SMOKE} --crash-at-step 20", check=False)
assert code == 3, f"expected the deliberate stop (exit 3), got exit {code}"
sh(f"ls {CKPT}/smoke-qlora")

**Paste back:** the `LoRA modules: ...` line, the `--crash-at-step` line, and the `ls` output.

## 6 · Resume the QLoRA run (~3 min)
Same command without the stop. It must say `resuming from checkpoint-20` and finish at step 30.

In [ ]:
%%time
sh(f"{TRAIN} --mode qlora --run-id smoke-qlora {SMOKE}")
s = json.load(open(f"{CKPT}/smoke-qlora/train_summary.json"))
assert s["resumed_from"] == "checkpoint-20" and s["global_step"] == 30, s
print(
    "RESUME OK:",
    {
        k: s[k]
        for k in (
            "resumed_from",
            "global_step",
            "steps_this_session",
            "train_loss",
            "final_val_loss",
            "peak_mem_gb",
        )
    },
)

**Paste back:** the `RESUME OK: ...` line.

## 7 · LoRA (bf16) smoke test, the main mode (~4 min)

In [ ]:
%%time
sh(f"{TRAIN} --mode lora --run-id smoke-lora {SMOKE}")
s = json.load(open(f"{CKPT}/smoke-lora/train_summary.json"))
print(
    "LORA OK:",
    {
        k: s[k]
        for k in (
            "global_step",
            "train_loss",
            "final_val_loss",
            "optimizer_step_s_median",
            "examples_per_s",
            "peak_mem_gb",
        )
    },
)

**Paste back:** the `LORA OK: ...` line.

## 8 · Memory and speed profile (~10–15 min)
3 image sizes × 3 per-device batch sizes, 8 timed steps each, bf16 LoRA with gradient checkpointing. Out-of-memory settings are recorded, not fatal.

In [ ]:
%%time
sh(
    f"{PY} -m vizwiz.train.sft profile --config {REPO_DIR}/configs/train/answer.yaml --mode lora --records {DATA}/processed --images {LOCAL}/images --budgets 262144 524288 1048576 --batch-sizes 1 2 4 --report {LOGS}/train_profile_l4.json"
)

**Paste back:** the nine `{'max_pixels': ...}` lines.

## 9 · Save the full package list

In [ ]:
sh(f"uv pip freeze --python {PY} > {LOGS}/freeze_train_l4.txt")
print("saved freeze_train_l4.txt")

## 10 · Flush Drive, then disconnect
Upload back from `MyDrive/vizwiz/`:
- `logs/train_profile_l4.json`
- `logs/freeze_train_l4.txt`
- `checkpoints/smoke-qlora/train_summary.json` (rename it `smoke_qlora_summary.json` if your browser would overwrite)
- `checkpoints/smoke-lora/train_summary.json` (or rename `smoke_lora_summary.json`)

Then Runtime → Disconnect and delete runtime. The smoke checkpoints on Drive (~1.5 GB) can be deleted afterwards: `MyDrive/vizwiz/checkpoints/smoke-*`.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")